In [ ]:
import pandas as pd

# Créer un jeu de commentaires fictifs (en français)
commentaires = [
    "Ce produit est génial, je le recommande vivement !",
    "Très mauvaise qualité, je suis déçu.",
    "J'adore, livraison rapide et produit parfait.",
    "Horrible expérience, je ne reviendrai plus jamais.",
    "Excellent service client, merci beaucoup !",
    "Produit cassé à l'arrivée, remboursement refusé.",
    "Super rapport qualité-prix, très satisfait.",
    "C'est nul, ça ne fonctionne pas du tout.",
    "Je suis ravi de mon achat, parfait !",
    "Arnaque totale, fuyez ce vendeur.",
    "Magnifique, exactement ce que je cherchais.",
    "Complètement raté, à éviter absolument.",
]

# 1 = positif, 0 = négatif
sentiments = [1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0]

# Créer le DataFrame
df = pd.DataFrame({
    "commentaire": commentaires,
    "sentiment": sentiments
})

print(df)


In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

# Créer le vectoriseur
vectoriseur = CountVectorizer()

# Transformer les commentaires en matrice de chiffres
X = vectoriseur.fit_transform(df["commentaire"])

# Afficher le résultat
print("Forme de la matrice :", X.shape)
print("\nLes mots du vocabulaire :")
print(vectoriseur.get_feature_names_out())

In [ ]:
from sklearn.linear_model import LogisticRegression

# Créer et entraîner le modèle
modele = LogisticRegression()
modele.fit(X, df["sentiment"])

# Vérifier qu'il a bien appris sur nos données
precision = modele.score(X, df["sentiment"])
print(f"Précision sur les données d'entraînement : {precision * 100:.2f}%")


In [ ]:
# Créer de nouveaux commentaires (que le modèle n'a jamais vus)
nouveaux_commentaires = [
    "J'adore ce produit, c'est parfait !",
    "Déçu, la qualité est mauvaise.",
    "Excellente expérience, je recommande !",
    "Arnaque, ne l'achetez pas.",
    "Produit correct mais rien d'exceptionnel."
]

# Transformer en chiffres (avec le même vectoriseur)
nouveaux_X = vectoriseur.transform(nouveaux_commentaires)

# Prédire le sentiment
predictions = modele.predict(nouveaux_X)
probabilites = modele.predict_proba(nouveaux_X)

# Afficher les résultats
for commentaire, pred, proba in zip(nouveaux_commentaires, predictions, probabilites):
    sentiment = "😊 POSITIF" if pred == 1 else "😠 NÉGATIF"
    confiance = max(proba) * 100
    print(f"\n💬 \"{commentaire}\"")
    print(f"   → {sentiment} (confiance : {confiance:.1f}%)")


In [ ]:
!pip install transformers torch -q

In [ ]:
from transformers import pipeline

# Charger un modèle pré-entraîné pour l'analyse de sentiment multilingue
# (Ça peut prendre 1-2 minutes pour le télécharger la première fois)
analyseur = pipeline(
    "sentiment-analysis",
    model="nlptown/bert-base-multilingual-uncased-sentiment"
)

print("✅ Modèle chargé avec succès !")


In [ ]:
# Tester avec les mêmes commentaires qu'avant
commentaires_test = [
    "J'adore ce produit, c'est parfait !",
    "Déçu, la qualité est mauvaise.",
    "Excellente expérience, je recommande !",
    "Arnaque, ne l'achetez pas.",
    "Produit correct mais rien d'exceptionnel.",
    "Pas mal du tout, je suis agréablement surpris !",
    "Je ne suis pas déçu, c'est vraiment bien."
]

# Analyser chaque commentaire
for commentaire in commentaires_test:
    resultat = analyseur(commentaire)[0]
    etoiles = resultat["label"]
    confiance = resultat["score"] * 100
    print(f"\n💬 \"{commentaire}\"")
    print(f"   → {etoiles} (confiance : {confiance:.1f}%)")


In [ ]:
import matplotlib.pyplot as plt

# Analyser tous les commentaires
resultats = [analyseur(c)[0] for c in commentaires_test]
notes = [int(r["label"].split()[0]) for r in resultats]

# Créer le graphique
plt.figure(figsize=(12, 6))
couleurs = ["red" if n <= 2 else "orange" if n == 3 else "green" for n in notes]
plt.barh(range(len(commentaires_test)), notes, color=couleurs)
plt.yticks(range(len(commentaires_test)), [c[:50] + "..." if len(c) > 50 else c for c in commentaires_test])
plt.xlabel("Note (étoiles)")
plt.title("Analyse de sentiment des commentaires")
plt.xlim(0, 5)
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()